# L11 · 통합 비교와 다음 연구

## Goal

**예상 시간:** 45분 · **경로:** fast, full

- 방법 선택 근거를 쓴다
- OPD2 delta를 계산한다
- avg@K와 pass@K를 분리한다

### 현재 위치: L10 → **L11** → finish

```text
Prompt/Data -> state source -> ... -> L11 -> ... -> fair evaluation
```

Alt text: The course map highlights L11 between its prerequisite and next lesson; every method remains connected to the same evaluation stage.

## Setup

In [1]:
LESSON_ID = "L11"
from pathlib import Path
import sys
import torch

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = Path.cwd().parents[1]
sys.path.insert(0, str(repo_root / "src"))

import opd_study
from opd_study.device import resolve_device
from opd_study.utils import seed_everything

seed_everything(42)
device_report = resolve_device("cpu")
print({"lesson": LESSON_ID, "opd_study": opd_study.__version__,
       "torch": torch.__version__, "device": device_report.selected,
       "profile": "toy", "network": "not required"})

{'lesson': 'L11', 'opd_study': '0.1.0.dev0', 'torch': '2.13.0', 'device': 'cpu', 'profile': 'toy', 'network': 'not required'}


## Steps

### 1/3 · 8–12 min

마지막에는 알고리즘 하나가 항상 최고라는 결론 대신 전제에 맞게 고른다. OPD2의 delta와 언어 유지, 작은 K 효율과 큰 K 능력 경계를 별도로 평가한다.

그림 대체 설명: 출력의 label과 숫자는 색 없이도 읽을 수 있다.

### 핵심 원리

OPD²는 post-trained teacher와 base teacher의 log-prob 차이를 이용해 post-training이 실제로 더 선호하게 만든 token에 집중한다. teacher 하나만 따라가는 OPD와 달리 `teacher - teacher_base` delta가 핵심이므로 두 model의 tokenizer/template 정합성이 필요하다.

test-time scaling에서는 `avg@K`(K개 sample의 평균 성공률)와 `pass@K`(하나라도 성공할 확률)를 분리한다. OPD가 작은 K 효율을 높이면서 큰 K에서 풀 수 있던 문제를 잃을 수도 있으므로 gained/lost solvability와 multilingual retention을 별도 평가한다.

### 실제 구현: 왜 이렇게 만들었나

OPD²는 teacher/base delta를 center하고 양의 개선 영역을 gate한다. test-time scaling helper는 boolean `[problem, sample]` 행렬을 받아 metric 정의를 코드에 고정한다. 이 둘은 같은 loss가 아니라 학습 선택과 평가 관점이다.

실제 코드: [`opd2.py`](../../src/opd_study/algorithms/opd2.py), [`test_time_scaling.py`](../../src/opd_study/diagnostics/test_time_scaling.py).

In [2]:
import inspect
from opd_study.algorithms import opd2_loss
from opd_study.diagnostics import scaling_metrics

objects_to_show = (opd2_loss, scaling_metrics,)
for object_to_show in objects_to_show:
    source_lines = inspect.getsource(object_to_show).splitlines()
    print(f"\n# {object_to_show.__module__}.{object_to_show.__qualname__}")
    print("\n".join(source_lines[:80]))
    if len(source_lines) > 80:
        print(f"... {len(source_lines) - 80} more lines; open the linked source file")


# opd_study.algorithms.opd2.opd2_loss
def opd2_loss(
    student_logits: Tensor,
    trajectories: TrajectoryBatch,
    teacher_signals: TeacherSignals,
    teacher_base_signals: TeacherSignals,
    *,
    centering_top_k: int | None = None,
) -> LossOutput:
    """Centered teacher-minus-base delta advantage with OPD-direction gating.

    This follows the paper's three essential choices: delta reward, action-independent
    centering, and the joint-sign condition.  It is intentionally not presented as a
    reproduction of the official large-scale TRL/GRPO recipe.
    """

    teacher = teacher_signals.logits
    teacher_base = teacher_base_signals.logits
    if teacher is None or teacher_base is None:
        raise ValueError("OPD² requires teacher and teacher-base logits")
    if teacher.shape != student_logits.shape or teacher_base.shape != student_logits.shape:
        raise ValueError("teacher, teacher-base and student logits must match")
    shifted_student, target_ids, _, pred

### 다른 선택지는 없나?

선택 규칙: single-turn이고 support가 좋으면 vanilla OPD부터, sampled variance가 크면 vOPD, long-horizon이면 TCOD/SOD/SAGE, post-training delta 보존이 목적이면 OPD²를 검토한다. 최종 선택은 같은 budget의 SFT/KD baseline과 held-out 평가로 결정한다.

### 2/3 · 실행하고 관찰하기

실행 전 예측: L11의 첫 출력에서 가장 먼저 확인해야 할 invariant는 무엇일까? 한 문장으로 적고 실행한다.

In [3]:
from opd_study.algorithms import opd2_loss
from opd_study.data import CharacterTokenizer, collate_examples, generate_tiny_arithmetic
from opd_study.types import TeacherSignals

tokenizer = CharacterTokenizer(); rows = generate_tiny_arithmetic(train_rows=2, validation_rows=1, test_rows=1).train
batch = collate_examples(rows, tokenizer)
shape = (*batch.token_ids.shape, tokenizer.vocab_size)
student = torch.randn(shape, requires_grad=True)
teacher, teacher_base = torch.randn(shape), torch.randn(shape)
delta_output = opd2_loss(student, batch, TeacherSignals(logits=teacher),
    TeacherSignals(logits=teacher_base), centering_top_k=16)
print("OPD2 gate rate:", delta_output.metrics["opd2/gate_rate"])
print("Delta isolates teacher post-training change; multilingual retention must be evaluated separately.")

OPD2 gate rate: 0.529411792755127
Delta isolates teacher post-training change; multilingual retention must be evaluated separately.


In [4]:
from opd_study.diagnostics.test_time_scaling import gained_and_lost_solvability, scaling_metrics

before = torch.tensor([[1,0,0,0], [0,0,1,0], [0,0,0,0]], dtype=torch.bool)
after = torch.tensor([[0,0,0,0], [1,1,0,0], [1,0,0,0]], dtype=torch.bool)
for k in (1, 2, 4):
    metric = scaling_metrics(after, k=k)
    print(k, "avg@K", metric.avg_at_k, "pass@K", metric.pass_at_k)
print("solvability:", gained_and_lost_solvability(before, after, k=4))

1 avg@K 0.6666666865348816 pass@K 0.6666666865348816
2 avg@K 0.5 pass@K 0.6666666865348816
4 avg@K 0.25 pass@K 0.6666666865348816
solvability: {'gained': 1, 'lost': 1, 'retained': 1, 'never_solved': 0}


## Checks

In [5]:
metric = scaling_metrics(after, k=4)
assert metric.avg_at_k != metric.pass_at_k
changes = gained_and_lost_solvability(before, after, k=4)
assert sum(changes.values()) == before.shape[0]
assert 0 <= delta_output.metrics["opd2/gate_rate"] <= 1
print("check passed: delta gating and capability-boundary accounting are explicit")

check passed: delta gating and capability-boundary accounting are explicit


**연습 (10분):** 방법 선택 memo를 5문장으로 쓴다: task horizon, teacher/student support, logit 접근성, hardware, SFT baseline을 반드시 포함하라.

<details><summary>확인 기준</summary>알고리즘 이름보다 전제와 측정 계획이 먼저 나오고 avg@K/pass@K 또는 언어 유지 중 관련 guardrail을 포함한다.</details>

## 내가 자주 틀리는 것

### M1 — avg@K와 pass@K를 바꿔 쓰기

- 틀린 형태: sample 평균 성공과 하나 이상 성공을 같은 수치로 쓴다.
- 왜 틀렸나: K가 커질 때 의미가 크게 갈린다.
- 고친 형태: sampling matrix에서 두 정의를 별도로 계산한다.
- 관련 검사: `test_avg_and_pass_at_k_are_not_interchangeable`

### M2 — 하나의 benchmark로 방법을 확정하기

- 틀린 형태: math accuracy만 보고 language retention과 큰-K 능력을 무시한다.
- 왜 틀렸나: post-training capability가 이동할 수 있다.
- 고친 형태: task, retention, support와 scaling guardrail을 함께 둔다.
- 관련 검사: `test_opd2_gate_closes_when_teacher_equals_base`

## 60초 요약

1. 방법 선택 근거를 쓴다
2. OPD2 delta를 계산한다
3. avg@K와 pass@K를 분리한다

## Next Steps

다음 노트북으로 가기 전, 위 assertion을 다시 실행하고 틀린 예측 한 줄을 남긴다.

### Sources

- [`opd2`](https://arxiv.org/abs/2607.15161v1) · `2607.15161v1` · license `CC-BY-4.0` · [audited manifest](../../docs/sources.yml)
- [`opd2_multilingual`](https://arxiv.org/abs/2608.05802v1) · `2608.05802v1` · license `CC-BY-4.0` · [audited manifest](../../docs/sources.yml)
- [`opd_test_time_scaling`](https://arxiv.org/abs/2608.11829v1) · `2608.11829v1` · license `arXiv-non-exclusive-distribution-1.0` · [audited manifest](../../docs/sources.yml)